# 02 · mT5-small (encoder–decoder) fine-tuning — English → Hindi

Foundations of Generative AI (ICT4403) — Assignment 3

**mT5** (Xue et al., 2021) is the multilingual T5: an encoder–decoder Transformer pretrained with span corruption on the
mC4 corpus covering 101 languages, Hindi included. We **fully fine-tune** `google/mt5-small` (300 M parameters,
250k-token SentencePiece vocabulary) on the **same 100k training pairs** that the from-scratch Transformer uses.

* Input: `"translate English to Hindi: <sentence>"`, target: the Hindi sentence (max 128 tokens each).
* AdamW, lr 5e-4, 500 warm-up steps then linear decay, batch 16, 3 epochs, best epoch chosen by validation loss.
* Training is in fp32, because mT5 is numerically unstable in fp16 (NaN losses).
* Decoding: greedy (main results) and temperature sampling (temperature study), as for every other model.

**Run:** Kaggle/Colab with a GPU and internet on → *Run all* (~45–60 min on a T4). Download `results_mt5.zip` at the end.

In [ ]:
!pip -q install sacrebleu sentencepiece

In [ ]:
import os, json, time, random, re
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SMOKE = os.environ.get("SMOKE") == "1"   # tiny CPU run used only to test the code
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_URL = os.environ.get("DATA_URL", "https://raw.githubusercontent.com/UtkarshKumarJha/GenAI-IA3/main/data/")
load = lambda name: pd.read_json(DATA_URL + name, lines=True)
train, valid, test, probes = (load(f) for f in ("train.jsonl", "valid.jsonl", "test.jsonl", "probes.jsonl"))
temp_set = test.iloc[:200]          # subset used for the sampling-temperature study
TEMPS = [0.5, 1.0, 1.5]             # main results use greedy decoding (T -> 0)
if SMOKE:
    train, valid, test, temp_set, probes = train[:200], valid[:16], test[:8], test[:4], probes[:4]
print(f"train {len(train):,} | valid {len(valid)} | test {len(test)} | probes {len(probes)} | device {DEVICE}",
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
def save_jsonl(df, path):
    df.to_json(path, orient="records", lines=True, force_ascii=False)

def run_all(translate, tag, out):
    """Greedy test translations (timed), temperature samples, and probe translations -> out/"""
    os.makedirs(out, exist_ok=True)
    t0 = time.perf_counter()
    hyps = translate(test.en.tolist())
    infer_s = time.perf_counter() - t0
    save_jsonl(test.assign(hyp=hyps), f"{out}/test_preds.jsonl")
    import sacrebleu
    bleu = sacrebleu.corpus_bleu(hyps, [test.hi.tolist()], tokenize="intl").score
    chrf = sacrebleu.corpus_chrf(hyps, [test.hi.tolist()]).score
    print(f"[{tag}] BLEU {bleu:.2f} | chrF {chrf:.2f} | {infer_s:.1f}s for {len(test)} sentences")
    for en, ref, hyp in list(zip(test.en, test.hi, hyps))[:3]:
        print(f"  EN : {en}\n  REF: {ref}\n  HYP: {hyp}\n")
    save_jsonl(probes.assign(hyp=translate(probes.en.tolist())), f"{out}/probe_preds.jsonl")
    return dict(infer_time_s=infer_s, infer_sent_per_s=len(test) / infer_s, quick_bleu=bleu, quick_chrf=chrf)

def run_temperatures(translate_sampled, out):
    rows = []
    for T in TEMPS:
        torch.manual_seed(SEED)
        for en, ref, hyp in zip(temp_set.en, temp_set.hi, translate_sampled(temp_set.en.tolist(), T)):
            rows.append(dict(temperature=T, en=en, hi=ref, hyp=hyp))
    save_jsonl(pd.DataFrame(rows), f"{out}/temperature_preds.jsonl")

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
print("transformers", transformers.__version__)
transformers.logging.set_verbosity_error()

MODEL_ID, PREFIX = "google/mt5-small", "translate English to Hindi: "
CFG = dict(batch=16, lr=5e-4, warmup=500, epochs=3, max_src=128, max_tgt=128)
if SMOKE: CFG.update(batch=4, warmup=5, epochs=1)
OUT = "results/mt5"; os.makedirs(OUT, exist_ok=True)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).float().to(DEVICE)
n_params = model.num_parameters()
print(f"parameters: {n_params / 1e6:.1f} M")
print(tok.tokenize(test.en[0])[:25]); print(tok.tokenize(test.hi[0])[:25])

In [ ]:
def encode_pairs(df):
    src = tok([PREFIX + s for s in df.en], truncation=True, max_length=CFG["max_src"])["input_ids"]
    tgt = tok(text_target=df.hi.tolist(), truncation=True, max_length=CFG["max_tgt"])["input_ids"]
    return list(zip(src, tgt))

def pad(seqs, value):
    out = torch.full((len(seqs), max(map(len, seqs))), value, dtype=torch.long)
    for i, s in enumerate(seqs): out[i, : len(s)] = torch.tensor(s)
    return out

def batches(pairs, shuffle):
    idx = np.random.permutation(len(pairs)) if shuffle else np.arange(len(pairs))
    chunk = CFG["batch"] * 100   # length-bucketing inside random chunks -> less padding
    for c in range(0, len(idx), chunk):
        part = sorted(idx[c: c + chunk], key=lambda i: len(pairs[i][0]))
        bs = [part[i: i + CFG["batch"]] for i in range(0, len(part), CFG["batch"])]
        if shuffle: random.shuffle(bs)
        for b in bs:
            src = pad([pairs[i][0] for i in b], tok.pad_token_id).to(DEVICE)
            yield dict(input_ids=src, attention_mask=(src != tok.pad_token_id).long(),
                       labels=pad([pairs[i][1] for i in b], -100).to(DEVICE))

train_pairs, valid_pairs = encode_pairs(train), encode_pairs(valid)
steps = CFG["epochs"] * -(-len(train_pairs) // CFG["batch"])
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, CFG["warmup"], steps)

def run_epoch(pairs, train_mode):
    model.train(train_mode); tot, n = 0.0, 0
    with torch.set_grad_enabled(train_mode):
        for b in batches(pairs, train_mode):
            loss = model(**b).loss
            if train_mode:
                opt.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            k = (b["labels"] != -100).sum().item(); tot += loss.item() * k; n += k
    return tot / n

history, best = [], float("inf")
print(f"validation loss before fine-tuning: {run_epoch(valid_pairs, False):.3f}")
t0 = time.perf_counter()
for ep in range(1, CFG["epochs"] + 1):
    te = time.perf_counter()
    tr, va = run_epoch(train_pairs, True), run_epoch(valid_pairs, False)
    history.append(dict(epoch=ep, train_loss=tr, val_loss=va, val_ppl=np.exp(va), sec=time.perf_counter() - te))
    print(f"epoch {ep} | train {tr:.3f} | val {va:.3f} (ppl {np.exp(va):.1f}) | {history[-1]['sec'] / 60:.1f} min")
    if va < best: best = va; torch.save(model.state_dict(), "mt5_best.pt")
train_time = time.perf_counter() - t0
model.load_state_dict(torch.load("mt5_best.pt"))
pd.DataFrame(history).to_csv(f"{OUT}/history.csv", index=False)
print(f"fine-tuning time {train_time / 60:.1f} min")

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history)
plt.figure(figsize=(6, 3.5)); plt.plot(h.epoch, h.train_loss, "o-", label="train"); plt.plot(h.epoch, h.val_loss, "o-", label="validation")
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.title("mT5-small fine-tuning"); plt.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/training_curve.png", dpi=120); plt.show()

## Translation (greedy and temperature sampling)

In [ ]:
@torch.no_grad()
def translate(sentences, temperature=0.0, bs=32, max_new=256):
    model.eval(); out = []
    kw = dict(do_sample=False, num_beams=1) if temperature == 0 else dict(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
    for i in range(0, len(sentences), bs):
        enc = tok([PREFIX + s for s in sentences[i: i + bs]], return_tensors="pt", padding=True,
                  truncation=True, max_length=CFG["max_src"]).to(DEVICE)
        gen = model.generate(**enc, max_new_tokens=max_new, **kw)
        out += [re.sub(r"<extra_id_\d+>", "", t).strip() for t in tok.batch_decode(gen, skip_special_tokens=True)]
    return out

stats = dict(model="mT5-small (fine-tuned)", architecture="encoder-decoder", adaptation="full fine-tuning",
             params=n_params, trainable_params=n_params, train_pairs=len(train), train_time_s=train_time,
             epochs=len(history), best_val_loss=best, tokenizer="SentencePiece unigram 250k (mC4, 101 languages)",
             avg_tokens_en=float(np.mean([len(x) for x in tok(test.en.tolist())["input_ids"]])),
             avg_tokens_hi=float(np.mean([len(x) for x in tok(test.hi.tolist())["input_ids"]])))
stats.update(run_all(translate, "mT5", OUT))
run_temperatures(lambda xs, T: translate(xs, temperature=T), OUT)
json.dump(stats, open(f"{OUT}/stats.json", "w"), indent=1)
stats

In [ ]:
import shutil; shutil.make_archive("results_mt5", "zip", ".", OUT)   # download this file